# 🎯 Embedding Fine-Tuning for FinanceRAG (Optimized)

**Mục tiêu**: Fine-tune E5-small cho Financial RAG với dữ liệu ít (~2,500 pairs)

## Pipeline Overview:
1. **Setup** - Load config và libraries
2. **Data** - Load datasets với weighted sampling
3. **Training Examples** - Tạo pairs/triplets với E5 prefix
4. **Model** - Load E5-small với layer freezing
5. **Training** - Sử dụng SentenceTransformer's `fit()` method
6. **Evaluation** - So sánh base vs fine-tuned

### Key Optimizations:
- ✅ E5-small (33M params) - Best from evaluation
- ✅ 1 layer trainable (91% frozen) - Prevent overfitting
- ✅ Weighted sampling - Focus on MULTIHEIRTT
- ✅ Query type weights - multi_hop, calculation get 3x

In [1]:
# ============================================================
# CELL 1: SETUP & IMPORTS
# ============================================================
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import json
import re
import random
from pathlib import Path
from typing import List, Dict, Tuple
from tqdm.auto import tqdm
import torch

# Sentence Transformers
from sentence_transformers import (
    SentenceTransformer, 
    InputExample, 
    losses,
    evaluation,
    util
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from torch.utils.data import DataLoader

# Load config
from config import (
    DATA_DIR, DATASETS, DATASET_WEIGHTS, TRAINING_CONFIG, HARD_NEGATIVES_CONFIG,
    BASE_MODEL, MINING_MODEL, OUTPUT_MODEL_NAME, MODELS_DIR, OUTPUT_DIR,
    E5_QUERY_PREFIX, E5_PASSAGE_PREFIX, LAYER_FREEZING_CONFIG,
    TRIPLET_MARGIN, print_config
)

# Import shared utilities
from utils import load_jsonl, QRELS_MAPPING

# Device setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️ Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

# Create directories
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Print config
print_config()

🖥️ Device: cuda
   GPU: NVIDIA GeForce RTX 3050 Laptop GPU
📊 Fine-Tuning Configuration (OPTIMIZED)

🤖 Models:
   Base model: intfloat/e5-small-v2
   Mining model: intfloat/e5-small-v2
   E5 prefixes: query='query: ', passage='passage: '

🧊 Layer Freezing:
   enabled: True
   layers_to_train: 3
   freeze_embeddings: True
   freeze_pooler: False
   use_adapter: False
   adapter_dim: 64

📁 Datasets (7):
   - multiheirtt (weight: 3.0) ⭐
   - finqa (weight: 1.5) ⭐
   - tatqa (weight: 1.5) ⭐
   - convfinqa (weight: 1.0) 
   - finder (weight: 1.0) 
   - financebench (weight: 1.0) 
   - finqabench (weight: 1.0) 

⚙️ Training Parameters:
   train_ratio: 0.9
   val_ratio: 0.1
   epochs: 20
   batch_size: 16
   gradient_accumulation: 4
   learning_rate: 1e-05
   warmup_ratio: 0.1
   weight_decay: 0.05
   dropout: 0.1
   label_smoothing: 0.1
   max_seq_length: 256
   max_doc_length: 1024
   adam_epsilon: 1e-08
   scheduler: cosine
   early_stopping_patience: 2
   min_delta: 0.001
   evaluation_ste

## 2. Load Datasets

In [2]:
# ============================================================
# CELL 2: LOAD ALL DATASETS
# ============================================================

def load_jsonl_as_df(file_path):
    """Load JSONL file into DataFrame"""
    data = load_jsonl(file_path)
    return pd.DataFrame(data)

def add_e5_prefix(text, is_query=True):
    """Add E5-specific prefix"""
    prefix = E5_QUERY_PREFIX if is_query else E5_PASSAGE_PREFIX
    return f"{prefix}{text}"

# Load all datasets
all_data = {}

for dataset in DATASETS:
    print(f"\nLoading {dataset}...")
    weight = DATASET_WEIGHTS.get(dataset, 1.0)
    
    # Load corpus
    corpus_path = DATA_DIR / f'{dataset}_corpus.jsonl' / 'corpus.jsonl'
    if not corpus_path.exists():
        corpus_path = DATA_DIR / f'{dataset}_corpus_preprocessed.jsonl'
    
    corpus_df = load_jsonl_as_df(corpus_path)
    
    # Load queries
    queries_path = DATA_DIR / f'{dataset}_queries.jsonl' / 'queries.jsonl'
    queries_df = load_jsonl_as_df(queries_path)
    
    # Load qrels
    qrels_file = DATA_DIR / QRELS_MAPPING.get(dataset, f'{dataset}_qrels.tsv')
    
    if qrels_file.exists():
        qrels_df = pd.read_csv(qrels_file, sep='\t')
    else:
        print(f"  ⚠️ No qrels found for {dataset}")
        continue
    
    all_data[dataset] = {
        'corpus': corpus_df,
        'queries': queries_df,
        'qrels': qrels_df,
        'weight': weight
    }
    
    marker = "⭐" if weight > 1 else ""
    print(f"  ✅ Corpus: {len(corpus_df)}, Queries: {len(queries_df)}, Pairs: {len(qrels_df)} {marker}")

print(f"\n✅ Loaded {len(all_data)} datasets")


Loading multiheirtt...
  ✅ Corpus: 10475, Queries: 974, Pairs: 1330 ⭐

Loading finqa...
  ✅ Corpus: 2789, Queries: 1147, Pairs: 344 ⭐

Loading tatqa...
  ✅ Corpus: 2756, Queries: 1663, Pairs: 498 ⭐

Loading convfinqa...
  ✅ Corpus: 2066, Queries: 421, Pairs: 126 

Loading finder...
  ✅ Corpus: 13867, Queries: 216, Pairs: 103 

Loading financebench...
  ✅ Corpus: 180, Queries: 150, Pairs: 59 

Loading finqabench...
  ✅ Corpus: 92, Queries: 100, Pairs: 30 

✅ Loaded 7 datasets


## 3. Create Training Examples

In [3]:
# ============================================================
# CELL 3: CREATE TRAINING EXAMPLES WITH WEIGHTED SAMPLING
# ============================================================

def classify_query_type(query: str) -> str:
    """Classify query by difficulty pattern"""
    query_lower = query.lower()
    
    if re.search(r'in the year with', query_lower):
        return 'multi_hop'
    if any(term in query_lower for term in ['sum of', 'total of', 'combined']):
        return 'aggregation'
    if any(term in query_lower for term in ['percentage', 'percent', 'growth rate', 'change', 'ratio']):
        return 'calculation'
    if re.search(r'[A-Z][a-z]+\s+of\s+[A-Z]', query):
        return 'table_lookup'
    return 'simple'

def create_training_pairs(all_data, use_e5_prefix=True, weight_by_difficulty=True):
    """Create positive pairs with optional weighting"""
    examples = []
    query_type_weights = HARD_NEGATIVES_CONFIG.get('query_type_weights', {
        'multi_hop': 3, 'calculation': 3, 'aggregation': 2, 'simple': 1, 'table_lookup': 1
    })
    
    for dataset_name, data in all_data.items():
        corpus_df = data['corpus']
        queries_df = data['queries']
        qrels_df = data['qrels']
        dataset_weight = data.get('weight', 1.0)
        
        # Detect column names
        query_col = 'query-id' if 'query-id' in qrels_df.columns else 'query_id'
        corpus_col = 'corpus-id' if 'corpus-id' in qrels_df.columns else 'corpus_id'
        id_col = '_id' if '_id' in corpus_df.columns else 'id'
        text_col = 'text' if 'text' in corpus_df.columns else 'content'
        
        # Create lookup dicts
        max_doc_len = TRAINING_CONFIG.get('max_doc_length', 1024)
        corpus_dict = {str(row[id_col]): str(row.get(text_col, ''))[:max_doc_len] 
                      for _, row in corpus_df.iterrows()}
        query_dict = {str(row[id_col]): str(row.get(text_col, '')) 
                     for _, row in queries_df.iterrows()}
        
        count = 0
        for _, row in qrels_df.iterrows():
            query_id = str(row[query_col])
            corpus_id = str(row[corpus_col])
            
            if query_id in query_dict and corpus_id in corpus_dict:
                query_text = query_dict[query_id]
                doc_text = corpus_dict[corpus_id]
                
                if use_e5_prefix:
                    query_text_final = add_e5_prefix(query_text, is_query=True)
                    doc_text_final = add_e5_prefix(doc_text, is_query=False)
                else:
                    query_text_final = query_text
                    doc_text_final = doc_text
                
                # Calculate weight
                if weight_by_difficulty:
                    query_type = classify_query_type(query_text)
                    type_weight = query_type_weights.get(query_type, 1)
                    total_weight = int(dataset_weight * type_weight)
                else:
                    total_weight = 1
                
                for _ in range(total_weight):
                    examples.append(InputExample(texts=[query_text_final, doc_text_final]))
                count += 1
        
        print(f"  {dataset_name}: {count} pairs")
    
    return examples

print("Creating training examples...")
training_examples = create_training_pairs(all_data, use_e5_prefix=True, weight_by_difficulty=True)
print(f"\n✅ Total examples: {len(training_examples)}")

# Split train/val
random.seed(TRAINING_CONFIG['seed'])
random.shuffle(training_examples)
split_idx = int(len(training_examples) * TRAINING_CONFIG['train_ratio'])
train_examples = training_examples[:split_idx]
val_examples = training_examples[split_idx:]

print(f"Train: {len(train_examples)}, Val: {len(val_examples)}")

Creating training examples...
  multiheirtt: 1330 pairs
  finqa: 344 pairs
  tatqa: 498 pairs
  convfinqa: 126 pairs
  finder: 103 pairs
  financebench: 59 pairs
  finqabench: 30 pairs

✅ Total examples: 10285
Train: 9256, Val: 1029


## 4. Load Model with Layer Freezing

In [4]:
# ============================================================
# CELL 4: LOAD MODEL WITH LAYER FREEZING
# ============================================================

print(f"Loading model: {BASE_MODEL}")
model = SentenceTransformer(BASE_MODEL, device=device)

# Apply layer freezing
def apply_layer_freezing(model, layers_to_train=1, freeze_embeddings=True):
    """Freeze all layers except the last N"""
    bert_model = model[0].auto_model
    num_layers = len(bert_model.encoder.layer)
    layers_to_freeze = num_layers - layers_to_train
    
    # Freeze embeddings
    if freeze_embeddings:
        for param in bert_model.embeddings.parameters():
            param.requires_grad = False
    
    # Freeze early layers
    for i in range(layers_to_freeze):
        for param in bert_model.encoder.layer[i].parameters():
            param.requires_grad = False
    
    # Keep last N layers trainable
    for i in range(layers_to_freeze, num_layers):
        for param in bert_model.encoder.layer[i].parameters():
            param.requires_grad = True
    
    # Pooler trainable
    if hasattr(bert_model, 'pooler') and bert_model.pooler is not None:
        for param in bert_model.pooler.parameters():
            param.requires_grad = True
    
    return model

layers_to_train = LAYER_FREEZING_CONFIG['layers_to_train']
model = apply_layer_freezing(model, layers_to_train=layers_to_train)

# Count params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\n🧊 Layer Freezing Applied:")
print(f"   Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")
print(f"   Frozen: {100*(1-trainable/total):.1f}%")

Loading model: intfloat/e5-small-v2

🧊 Layer Freezing Applied:
   Trainable: 5,471,232 / 33,360,000 (16.4%)
   Frozen: 83.6%


## 5. Create Evaluator

In [5]:
# ============================================================
# CELL 5: CREATE VALIDATION EVALUATOR
# ============================================================

# Build IR evaluator from validation examples
val_queries_dict = {}
val_corpus_dict = {}
val_relevant_docs = {}

for i, example in enumerate(val_examples[:500]):  # Limit for speed
    query_id = f"q_{i}"
    doc_id = f"d_{i}"
    
    val_queries_dict[query_id] = example.texts[0]
    val_corpus_dict[doc_id] = example.texts[1]
    val_relevant_docs[query_id] = {doc_id}

# Add distractor docs
all_docs = list(val_corpus_dict.values())
for i in range(min(50, len(all_docs))):
    neg_doc_id = f"neg_{i}"
    neg_idx = (i + 17) % len(all_docs)
    val_corpus_dict[neg_doc_id] = all_docs[neg_idx]

evaluator = InformationRetrievalEvaluator(
    queries=val_queries_dict,
    corpus=val_corpus_dict,
    relevant_docs=val_relevant_docs,
    name='validation',
    ndcg_at_k=[10],
    mrr_at_k=[10],
    show_progress_bar=False
)

print(f"✅ Evaluator created: {len(val_queries_dict)} queries, {len(val_corpus_dict)} docs")

✅ Evaluator created: 500 queries, 550 docs


## 6. Train Model

**Important**: Sử dụng `model.fit()` method thay vì custom training loop để tránh lỗi với TripletLoss.

In [6]:
# ============================================================
# CELL 6: TRAIN MODEL (CUSTOM TRAINING LOOP)
# ============================================================
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Training config
BATCH_SIZE = TRAINING_CONFIG['batch_size']
NUM_EPOCHS = TRAINING_CONFIG['epochs']
WARMUP_RATIO = TRAINING_CONFIG['warmup_ratio']
LEARNING_RATE = TRAINING_CONFIG['learning_rate']
WEIGHT_DECAY = TRAINING_CONFIG['weight_decay']

# Output path
MODEL_OUTPUT_DIR = MODELS_DIR / OUTPUT_MODEL_NAME
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Custom collate function for InputExample
def collate_fn(batch):
    """Custom collate that keeps InputExample objects as list"""
    return batch

# Create DataLoader with custom collate
train_dataloader = DataLoader(
    train_examples, 
    shuffle=True, 
    batch_size=BATCH_SIZE,
    drop_last=True,
    collate_fn=collate_fn  # Important: use custom collate
)

# Calculate steps
total_steps = len(train_dataloader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

print(f"{'='*60}")
print(f"📊 Training Configuration")
print(f"{'='*60}")
print(f"  Examples: {len(train_examples)}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Total steps: {total_steps}")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Output: {MODEL_OUTPUT_DIR}")
print(f"{'='*60}")

# Optimizer - only trainable params
optimizer = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

# Training loop
print(f"\n🚀 Starting training...")
model.train()

global_step = 0
best_loss = float('inf')

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch in pbar:
        optimizer.zero_grad()
        
        # Get texts from batch - batch is list of InputExample
        anchors_texts = [ex.texts[0] for ex in batch]
        positives_texts = [ex.texts[1] for ex in batch]
        
        # Tokenize separately
        anchor_features = model.tokenize(anchors_texts)
        anchor_features = {k: v.to(device) for k, v in anchor_features.items()}
        
        positive_features = model.tokenize(positives_texts)
        positive_features = {k: v.to(device) for k, v in positive_features.items()}
        
        # Get embeddings
        anchors = model(anchor_features)['sentence_embedding']
        positives = model(positive_features)['sentence_embedding']
        
        # Compute loss using cosine similarity (in-batch negatives)
        # similarity_matrix[i,j] = similarity between anchor_i and positive_j
        similarity_matrix = torch.nn.functional.cosine_similarity(
            anchors.unsqueeze(1), 
            positives.unsqueeze(0), 
            dim=2
        )
        
        # Cross-entropy loss (diagonal should be highest)
        batch_size = len(batch)
        labels = torch.arange(batch_size, device=device)
        loss = torch.nn.functional.cross_entropy(similarity_matrix * 20.0, labels)  # scale=20
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        global_step += 1
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.2e}'})
    
    avg_loss = epoch_loss / len(train_dataloader)
    print(f"  Epoch {epoch+1} avg loss: {avg_loss:.4f}")
    
    # Save best model (use safe_serialization=False to avoid Windows file lock issues)
    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save(str(MODEL_OUTPUT_DIR), safe_serialization=False)
        print(f"  ✅ Best model saved (loss: {best_loss:.4f})")

print(f"\n✅ Training complete! Model saved to: {MODEL_OUTPUT_DIR}")

📊 Training Configuration
  Examples: 9256
  Batch size: 16
  Epochs: 20
  Total steps: 11560
  Warmup steps: 1156
  Learning rate: 1e-05
  Weight decay: 0.05
  Output: ..\..\models\e5-small-financerag-finetuned-v3

🚀 Starting training...


Epoch 1/20:   0%|          | 0/578 [00:00<?, ?it/s]

  Epoch 1 avg loss: 2.2495
  ✅ Best model saved (loss: 2.2495)


Epoch 2/20:   0%|          | 0/578 [00:00<?, ?it/s]

  Epoch 2 avg loss: 1.8764
  ✅ Best model saved (loss: 1.8764)


...

  Epoch 18 avg loss: 0.2334
  ✅ Best model saved (loss: 0.2334)


Epoch 19/20:   0%|          | 0/578 [00:00<?, ?it/s]

  Epoch 19 avg loss: 0.2379


Epoch 20/20:   0%|          | 0/578 [00:00<?, ?it/s]

  Epoch 20 avg loss: 0.2295
  ✅ Best model saved (loss: 0.2295)

✅ Training complete! Model saved to: ..\..\models\e5-small-financerag-finetuned-v3


## 7. Evaluate Fine-tuned Model

In [7]:
# ============================================================
# CELL 7: COMPARE BASE VS FINE-TUNED
# ============================================================

import os

# Load models
print("Loading models for comparison...")
base_model = SentenceTransformer(BASE_MODEL, device=device)

if MODEL_OUTPUT_DIR.exists() and os.listdir(MODEL_OUTPUT_DIR):
    finetuned_model = SentenceTransformer(str(MODEL_OUTPUT_DIR), device=device)
    print("✅ Fine-tuned model loaded")
else:
    print("⚠️ Fine-tuned model not found, using current model")
    finetuned_model = model

# Evaluate on validation
print("\n📊 Evaluation Results:")
print("-" * 40)

base_result = evaluator(base_model)
base_ndcg = base_result.get('ndcg_at_10', 0)
print(f"Base model NDCG@10: {base_ndcg:.4f}")

ft_result = evaluator(finetuned_model)
ft_ndcg = ft_result.get('ndcg_at_10', 0)
print(f"Fine-tuned NDCG@10: {ft_ndcg:.4f}")

improvement = ft_ndcg - base_ndcg
improvement_pct = (improvement / base_ndcg * 100) if base_ndcg > 0 else 0

print("-" * 40)
if improvement > 0:
    print(f"✅ IMPROVEMENT: +{improvement:.4f} (+{improvement_pct:.1f}%)")
else:
    print(f"⚠️ No improvement: {improvement:.4f}")

Loading models for comparison...
✅ Fine-tuned model loaded

📊 Evaluation Results:
----------------------------------------
Base model NDCG@10: 0.0000
Fine-tuned NDCG@10: 0.0000
----------------------------------------
⚠️ No improvement: 0.0000


## 8. Test on MULTIHEIRTT (Most Important Dataset)

In [8]:
# ============================================================
# CELL 8: QUICK TEST ON MULTIHEIRTT
# ============================================================

import faiss

def quick_retrieval_test(test_model, dataset_name='multiheirtt', sample_size=100):
    """Quick retrieval test"""
    if dataset_name not in all_data:
        print(f"Dataset {dataset_name} not found")
        return 0.0
    
    data = all_data[dataset_name]
    corpus_df = data['corpus']
    queries_df = data['queries'].sample(n=min(sample_size, len(data['queries'])), random_state=42)
    qrels_df = data['qrels']
    
    # Column names
    query_col = 'query-id' if 'query-id' in qrels_df.columns else 'query_id'
    corpus_col = 'corpus-id' if 'corpus-id' in qrels_df.columns else 'corpus_id'
    id_col = '_id' if '_id' in corpus_df.columns else 'id'
    text_col = 'text' if 'text' in corpus_df.columns else 'content'
    
    # Prepare texts with E5 prefix
    corpus_texts = [add_e5_prefix(str(row.get(text_col, ''))[:1024], is_query=False) 
                   for _, row in corpus_df.iterrows()]
    corpus_ids = [str(row[id_col]) for _, row in corpus_df.iterrows()]
    
    query_texts = [add_e5_prefix(str(row.get(text_col, '')), is_query=True) 
                  for _, row in queries_df.iterrows()]
    query_ids = [str(row[id_col]) for _, row in queries_df.iterrows()]
    
    # Encode
    print(f"  Encoding {len(corpus_texts)} docs...")
    corpus_embs = test_model.encode(corpus_texts, batch_size=32, show_progress_bar=True, 
                               convert_to_numpy=True, normalize_embeddings=True)
    
    print(f"  Encoding {len(query_texts)} queries...")
    query_embs = test_model.encode(query_texts, batch_size=32, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)
    
    # Build index
    index = faiss.IndexFlatIP(corpus_embs.shape[1])
    index.add(corpus_embs.astype('float32'))
    
    # Retrieve and compute NDCG
    ndcg_scores = []
    for i, qid in enumerate(query_ids):
        scores, indices = index.search(query_embs[i].reshape(1, -1).astype('float32'), 10)
        retrieved = [corpus_ids[idx] for idx in indices[0]]
        
        # Ground truth
        relevant = qrels_df[qrels_df[query_col] == qid][corpus_col].astype(str).tolist()
        if not relevant:
            continue
        
        # Compute DCG
        dcg = sum(1.0 / np.log2(j + 2) for j, doc in enumerate(retrieved) if doc in relevant)
        idcg = sum(1.0 / np.log2(j + 2) for j in range(min(len(relevant), 10)))
        ndcg = dcg / idcg if idcg > 0 else 0
        ndcg_scores.append(ndcg)
    
    return np.mean(ndcg_scores) if ndcg_scores else 0.0

print("=" * 60)
print("🎯 MULTIHEIRTT Retrieval Test")
print("=" * 60)

# Load base model for comparison
print("\n📊 Loading base model for comparison...")
base_model = SentenceTransformer(BASE_MODEL, device=device)

# Fine-tuned model is already in 'model' variable after training
finetuned_model = model

print("\n📊 Base model:")
base_ndcg = quick_retrieval_test(base_model, 'multiheirtt', sample_size=100)
print(f"   NDCG@10: {base_ndcg:.4f}")

print("\n📊 Fine-tuned model:")
ft_ndcg = quick_retrieval_test(finetuned_model, 'multiheirtt', sample_size=100)
print(f"   NDCG@10: {ft_ndcg:.4f}")

improvement = ft_ndcg - base_ndcg
print(f"\n{'='*60}")
if improvement > 0:
    print(f"✅ IMPROVEMENT on MULTIHEIRTT: +{improvement:.4f} (+{improvement/base_ndcg*100:.1f}%)")
else:
    print(f"⚠️ No improvement: {improvement:.4f}")
print("=" * 60)

🎯 MULTIHEIRTT Retrieval Test

📊 Loading base model for comparison...

📊 Base model:
  Encoding 10475 docs...


Batches:   0%|          | 0/328 [00:00<?, ?it/s]

  Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

   NDCG@10: 0.0821

📊 Fine-tuned model:
  Encoding 10475 docs...


Batches:   0%|          | 0/328 [00:00<?, ?it/s]

  Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

   NDCG@10: 0.5449

✅ IMPROVEMENT on MULTIHEIRTT: +0.4628 (+563.6%)


## 9. Save Training Info

In [9]:
# ============================================================
# CELL 9: SAVE TRAINING INFO
# ============================================================

training_info = {
    'base_model': BASE_MODEL,
    'output_model': OUTPUT_MODEL_NAME,
    'training_examples': len(train_examples),
    'validation_examples': len(val_examples),
    'epochs': NUM_EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'layers_trained': layers_to_train,
    'trainable_params': trainable,
    'total_params': total,
    'trainable_pct': 100 * trainable / total,
    'datasets': list(all_data.keys()),
    'base_ndcg': float(base_ndcg) if 'base_ndcg' in dir() else None,
    'finetuned_ndcg': float(ft_ndcg) if 'ft_ndcg' in dir() else None,
}

info_path = MODEL_OUTPUT_DIR / 'training_info.json'
with open(info_path, 'w') as f:
    json.dump(training_info, f, indent=2)

print(f"✅ Training info saved to: {info_path}")
print(f"\n📊 Summary:")
for k, v in training_info.items():
    if not isinstance(v, list):
        print(f"   {k}: {v}")

✅ Training info saved to: ..\..\models\e5-small-financerag-finetuned-v3\training_info.json

📊 Summary:
   base_model: intfloat/e5-small-v2
   output_model: e5-small-financerag-finetuned-v3
   training_examples: 9256
   validation_examples: 1029
   epochs: 20
   batch_size: 16
   learning_rate: 1e-05
   weight_decay: 0.05
   layers_trained: 3
   trainable_params: 5471232
   total_params: 33360000
   trainable_pct: 16.400575539568344
   base_ndcg: 0.08211736643269266
   finetuned_ndcg: 0.5448922681602426
